# TrpB stability plots

These figures use the same plotting functions as the PDZ notebook. They read only saved `metadata.json`, `summary.csv`, and `overlaps.csv`; they do not reopen `stability.nc` or any HMC fit.

Individual reports and joint comparisons are rendered separately for `log_mean_growth` and posterior-predictive `count_ratio`. Four limited Exponential-Exponential source artifacts lack posterior-predictive counts, so they remain in log-growth comparisons but are omitted from count-ratio comparisons. Joint figures summarize calibration ECDFs and signed 5–95% posterior-rank divergence across the available fits.

## Configuration

The featured examples use the Exponential-Exponential fit for libA, libE, and libI. Because the limited libE artifact lacks posterior-predictive counts, the featured count-ratio comparison contains libA and libI; the all-fit count-ratio comparison still spans every library through 14 supported fits.

In [ ]:
# ruff: noqa: E402
# pyright: reportMissingImports=false

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "dms-fitness-stability"):
            if (candidate / "src" / "dms_fitness_stability").is_dir():
                return candidate
    raise FileNotFoundError("Could not find the dms-fitness-stability repository.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dms_fitness_stability import (
    plot_calibration_divergence,
    plot_calibration_overlay,
    plot_rank_divergence_boxplots,
    plot_report,
)

DATA_ROOT = Path(os.environ.get("DMS_FITNESS_STABILITY_DATA", "/data/data_dms_stability"))
RESULT_ROOT = DATA_ROOT / "trpb" / "results"
SCORES = ("log_mean_growth", "count_ratio")
SCORE_TITLES = {
    "log_mean_growth": "Log mean growth",
    "count_ratio": "Count ratio",
}
K = 50
INDIVIDUAL_REPORTED_TOP_N = 50
RANK_DIVERGENCE_COHORTS = (100, 1000)

LIBRARY_ORDER = tuple(f"lib{letter}" for letter in "ABCDEFGHI")
MODEL_ORDER = (
    "Exponential-Exponential",
    "Exponential-Sigmoid",
    "Gamma-Exponential",
)
MODEL_LABELS = {
    "Exponential-Exponential": "Exp–Exp",
    "Exponential-Sigmoid": "Exp–Sigmoid",
    "Gamma-Exponential": "Gamma–Exp",
}


def fit_label(library: str, model: str, status: str) -> str:
    suffix = " *" if status == "limited" else ""
    return f"{library} · {MODEL_LABELS[model]}{suffix}"


fit_records = []
for metadata_path in RESULT_ROOT.glob("*/*/metadata.json"):
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    available_scores = tuple(score for score in SCORES if score in metadata["scores"])
    if not available_scores:
        raise ValueError(f"{metadata_path.parent} has none of the requested scores.")
    fit_records.append(
        {
            "library": str(metadata["library"]),
            "model": str(metadata["model"]),
            "artifact_status": str(metadata["artifact_status"]),
            "artifact_issues": tuple(metadata["artifact_issues"]),
            "available_scores": available_scores,
            "result_dir": metadata_path.parent,
        }
    )

fit_records.sort(
    key=lambda row: (
        LIBRARY_ORDER.index(row["library"]),
        MODEL_ORDER.index(row["model"]),
    )
)
if len(fit_records) != 18:
    raise ValueError(f"Expected 18 TrpB results, found {len(fit_records)}.")

ALL_RESULTS = [
    (
        fit_label(row["library"], row["model"], row["artifact_status"]),
        row["result_dir"],
    )
    for row in fit_records
]
AVAILABLE_SCORES = {
    label: row["available_scores"]
    for (label, _), row in zip(ALL_RESULTS, fit_records)
}
RESULTS_BY_SCORE = {
    score: [
        (label, result_dir)
        for label, result_dir in ALL_RESULTS
        if score in AVAILABLE_SCORES[label]
    ]
    for score in SCORES
}

FEATURED_KEYS = {
    ("libA", "Exponential-Exponential"),
    ("libE", "Exponential-Exponential"),
    ("libI", "Exponential-Exponential"),
}
FEATURED_RESULTS = [
    (
        fit_label(row["library"], row["model"], row["artifact_status"]),
        row["result_dir"],
    )
    for row in fit_records
    if (row["library"], row["model"]) in FEATURED_KEYS
]
if len(FEATURED_RESULTS) != 3:
    raise ValueError("Expected exactly three featured results.")
FEATURED_RESULTS_BY_SCORE = {
    score: [
        (label, result_dir)
        for label, result_dir in FEATURED_RESULTS
        if score in AVAILABLE_SCORES[label]
    ]
    for score in SCORES
}

print(f"Result root: {RESULT_ROOT}")
print(
    f"Loaded {len(ALL_RESULTS)} fits; featured: "
    f"{', '.join(label for label, _ in FEATURED_RESULTS)}"
)
for score in SCORES:
    print(
        f"{SCORE_TITLES[score]}: {len(RESULTS_BY_SCORE[score])} all-fit results; "
        f"{len(FEATURED_RESULTS_BY_SCORE[score])} featured results"
    )
missing_count_ratio = [
    label for label, _ in ALL_RESULTS if "count_ratio" not in AVAILABLE_SCORES[label]
]
if missing_count_ratio:
    print("Count ratio unavailable: " + ", ".join(missing_count_ratio))
print("* denotes a limited source artifact; see KNOWN_ISSUES.md.")

## Three featured results

Each call below uses the same `plot_report` interface as the PDZ notebook. Separate rank, posterior-value, and rank-calibration views are requested for every available score; the limited libE artifact renders only `log_mean_growth`.

In [ ]:
from matplotlib.figure import Figure


def display_report(label: str, result_dir: Path) -> dict[str, Figure]:
    display(Markdown(f"### {label}"))
    figures = plot_report(
        result_dir,
        k=K,
        scores=AVAILABLE_SCORES[label],
        views=("rank", "value", "calibration"),
        reported_top_n=INDIVIDUAL_REPORTED_TOP_N,
    )
    for figure in figures.values():
        display(figure)
        plt.close(figure)
    return figures


featured_reports = {
    label: display_report(label, result_dir)
    for label, result_dir in FEATURED_RESULTS
}

## Joint reported-rank calibration

These are directional reported-rank percentile ECDFs, the rank-calibration quantity stored by the analysis. They are not quantiles of published fitness on the posterior score scale; those scales are not directly interchangeable.

Separate panel sets compare `log_mean_growth` and `count_ratio`. For each score, the first overlay highlights supported featured fits, the second overlays every supported fit, and the divergence plot reduces each all-fit curve to its area from the uniform diagonal.

In [ ]:
featured_calibration_figures = {}
featured_calibration_tables = {}
all_calibration_figures = {}
all_calibration_tables = {}
calibration_divergence_figures = {}
calibration_divergence_tables = {}

for score in SCORES:
    display(Markdown(f"### {SCORE_TITLES[score]}"))
    featured_results = FEATURED_RESULTS_BY_SCORE[score]
    all_results = RESULTS_BY_SCORE[score]

    figure, _, featured_calibration = plot_calibration_overlay(
        featured_results,
        score=score,
        k=K,
        cohort="tracked",
        alpha=0.85,
    )
    featured_calibration_figures[score] = figure
    featured_calibration_tables[score] = featured_calibration
    display(figure)
    plt.close(figure)

    figure, _, all_calibration = plot_calibration_overlay(
        all_results,
        score=score,
        k=K,
        cohort="tracked",
        alpha=0.45,
    )
    all_calibration_figures[score] = figure
    all_calibration_tables[score] = all_calibration
    display(figure)
    plt.close(figure)

    figure, _, calibration_divergence = plot_calibration_divergence(
        all_results,
        score=score,
        k=K,
        cohort="tracked",
    )
    calibration_divergence_figures[score] = figure
    calibration_divergence_tables[score] = calibration_divergence
    display(figure)
    plt.close(figure)

    display(
        calibration_divergence.sort_values("calibration_l1_area").reset_index(drop=True)
    )

## Signed posterior-rank divergence

For each available analysis and score, lower deviation is posterior 5th-percentile rank minus reported rank and upper deviation is posterior 95th-percentile rank minus reported rank. Zero indicates agreement with the reported rank. Separate `log_mean_growth` and `count_ratio` panels compare the reported top 100 with the stored reported top 1,000 cohort.

In [ ]:
rank_divergence_figures = {score: {} for score in SCORES}
rank_divergence_tables = {score: {} for score in SCORES}

for score in SCORES:
    display(Markdown(f"### {SCORE_TITLES[score]}"))

    for reported_top_n in RANK_DIVERGENCE_COHORTS:
        figure, _, divergence = plot_rank_divergence_boxplots(
            RESULTS_BY_SCORE[score],
            score=score,
            k=K,
            reported_top_n=reported_top_n,
        )
        rank_divergence_figures[score][reported_top_n] = figure
        rank_divergence_tables[score][reported_top_n] = divergence
        display(figure)
        plt.close(figure)

## Individual reports for all 18 fits

The same PDZ-style report is applied to every TrpB fit in library/model order. Four limited Exponential-Exponential artifacts render only `log_mean_growth`; the other 14 render both `log_mean_growth` and `count_ratio`. Asterisks mark source artifacts that should be regenerated before publication-quality reporting.

In [ ]:
all_reports = {}

for label, result_dir in ALL_RESULTS:
    all_reports[label] = display_report(label, result_dir)


print(f"Rendered {len(all_reports)} individual reports.")